# Vision Tracker Evaluation — RT-DETR + ByteTrack

Runs the RT-DETR model on all vision MOT sequences, compares predictions to GT, 
and reports standard MOT metrics.

**Model:** `rt_detr_solaqua_fish_120e_fair/weights/best.pt`  
**Class:** fish (class_id = 1)  
**Metric matching:** IoU ≥ 0.5

Run from `tracking/` directory.

In [ ]:
import sys
sys.path.insert(0, "..")  # make repo root importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from tracking.utils.mot_metrics import (
    load_gt, run_inference, img_dir_for, frame_map,
    compute_metrics, format_summary, build_summary_table,
    plot_metrics_bar, plot_error_breakdown,
    plot_track_comparison, plot_det_timeline, plot_id_switches,
)

plt.rcParams.update({"figure.dpi": 130, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
REPO        = Path("..").resolve()
VISION_ROOT = REPO / "data-processing" / "vision" / "MOT"
MODEL_PATH  = REPO / "runs/detect/outputs/training/solaqua_fish/rt_detr_solaqua_fish_120e_fair/weights/best.pt"

# ── Inference settings ─────────────────────────────────────────────────────────
CONF       = 0.25
IOU_NMS    = 0.45   # NMS threshold for detector
IOU_MATCH  = 0.5    # IoU threshold for GT↔pred matching
DEVICE     = "cuda:0"
FRAME_RATE = 30

SEQUENCES = sorted([p.name for p in VISION_ROOT.iterdir() if p.is_dir()])
print(f"Model   : {MODEL_PATH.name}")
print(f"Device  : {DEVICE}")
print(f"Sequences ({len(SEQUENCES)}):")
for s in SEQUENCES:
    print(f"  {s}")

## 1  Run inference on all sequences
Results are cached in memory. Re-run this cell to refresh.

In [ ]:
gt_dfs   = {}   # {seq_name: gt DataFrame (fish only)}
pred_dfs = {}   # {seq_name: pred DataFrame (fish only)}
total_frames = {}

for seq in SEQUENCES:
    seq_path = VISION_ROOT / seq
    idir     = img_dir_for(seq_path, "vision")

    # Ground truth — fish only (class_id = 1)
    gt = load_gt(seq_path / "gt" / "gt.txt", idir, class_id=1)

    # Inference
    pred = run_inference(MODEL_PATH, idir,
                         conf=CONF, iou=IOU_NMS,
                         device=DEVICE, frame_rate=FRAME_RATE,
                         desc=seq)
    pred_fish = pred[pred["class_id"] == 1].copy()

    gt_dfs[seq]   = gt
    pred_dfs[seq] = pred_fish
    total_frames[seq] = len(list(idir.glob("*.jpg")))
    print(f"{seq}: GT={len(gt)} annots, Pred={len(pred_fish)} annots")

## 2  Compute metrics

In [ ]:
accs    = {}
results = {}

for seq in SEQUENCES:
    acc, summary = compute_metrics(gt_dfs[seq], pred_dfs[seq], iou_threshold=IOU_MATCH)
    accs[seq]    = acc
    results[seq] = summary
    print(f"{seq[-8:]}  MOTA={float(summary['mota'].iloc[0])*100:.1f}%  "
          f"IDF1={float(summary['idf1'].iloc[0])*100:.1f}%  "
          f"MOTP={float(summary['motp'].iloc[0])*100:.1f}%  "
          f"IDSW={int(summary['num_switches'].iloc[0])}")

print()
display(build_summary_table(results))

## 3  Metrics bar charts

In [ ]:
fig = plot_metrics_bar(results, metrics=("MOTA", "IDF1", "MOTP", "Recall", "Precision"),
                       title="Vision — RT-DETR + ByteTrack")
plt.savefig("outputs/vision_metrics_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## 4  Error breakdown — FP / FN / ID switches

In [ ]:
fig = plot_error_breakdown(results, title="Vision — error breakdown (normalised by GT)")
plt.savefig("outputs/vision_error_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 5  GT vs predicted track count

In [ ]:
fig = plot_track_comparison(gt_dfs, pred_dfs)
plt.savefig("outputs/vision_track_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6  Detection count per frame — GT vs predicted

In [ ]:
fig = plot_det_timeline(gt_dfs, pred_dfs, total_frames,
                        title="Vision — GT vs predicted fish detections per frame")
plt.savefig("outputs/vision_det_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## 7  Cumulative ID switches over time

In [ ]:
fig = plot_id_switches(accs, total_frames,
                       title="Vision — cumulative ID switches per frame")
plt.savefig("outputs/vision_id_switches.png", dpi=150, bbox_inches="tight")
plt.show()

## 8  MT / PT / ML breakdown
Mostly Tracked (>80% life covered), Partially Tracked, Mostly Lost (<20%).

In [ ]:
seq_names = list(results.keys())
mt_vals = [int(results[s]["mostly_tracked"].iloc[0])   for s in seq_names]
pt_vals = [int(results[s]["partially_tracked"].iloc[0]) for s in seq_names]
ml_vals = [int(results[s]["mostly_lost"].iloc[0])       for s in seq_names]

x = np.arange(len(seq_names))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x, mt_vals, label="MT (>80%)",  color="#2E7D32", edgecolor="white")
ax.bar(x, pt_vals, bottom=mt_vals, label="PT (20–80%)", color="#FFA726", edgecolor="white")
ax.bar(x, ml_vals, bottom=[m+p for m,p in zip(mt_vals,pt_vals)],
       label="ML (<20%)", color="#EF5350", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([s[-8:] for s in seq_names], rotation=12, ha="right", fontsize=8)
ax.set_ylabel("Number of GT tracks")
ax.set_title("Vision — Mostly / Partially / Mostly-Lost tracks", fontweight="bold")
ax.legend(fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/vision_mt_pt_ml.png", dpi=150, bbox_inches="tight")
plt.show()